In [ ]:
import numpy as np

import numpy.linalg as la

import matplotlib.pyplot as plt

import cv2

from PIL import Image

import pickle

import os

In [ ]:
DATA_DIR = './cifar-10-batches-py'

# Load a single CIFAR-10 batch from disk
def load_batch(filepath):

    with open(filepath, 'rb') as f:

        d = pickle.load(f, encoding='bytes')

    images = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

    labels = np.array(d[b'labels'])

    return images, labels

# Load human-readable class names from batches.meta
def load_class_names(meta_filepath):

    with open(meta_filepath, 'rb') as f:

        meta = pickle.load(f, encoding='bytes')

    class_names = [name.decode('utf-8') for name in meta[b'label_names']]

    return class_names

# Load all 5 training batches and concatenate
train_batches = [load_batch(os.path.join(DATA_DIR, f'data_batch_{i}')) for i in range(1, 6)]

train_images = np.concatenate([b[0] for b in train_batches])

train_labels = np.concatenate([b[1] for b in train_batches])

# Load test batch
test_images, test_labels = load_batch(os.path.join(DATA_DIR, 'test_batch'))

# Load class names
class_names = load_class_names(os.path.join(DATA_DIR, 'batches.meta'))

print(len(train_images), len(test_images), train_images[0].shape, class_names)

In [ ]:
# SIFT keypoint extraction
# Upscale to 128x128 so the Gaussian scale space has enough room to detect features.
# Low contrastThreshold and edgeThreshold to recover keypoints on small CIFAR images.
# Returns: float32 array of shape (N_keypoints, 128), or None if no keypoints found.
sift = cv2.SIFT_create(contrastThreshold=0.02, edgeThreshold=5)

def extract_sift(img_rgb):

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    img_up = cv2.resize(gray, (128, 128))

    _, descs = sift.detectAndCompute(img_up, None)

    return descs

# Vocabulary building: sample 5000 training images, collect all SIFT descriptors,
# then run k-means to produce a K-word visual codebook.
K = 500

N_VOCAB_IMAGES = 5000

np.random.seed(42)

vocab_indices = np.random.choice(len(train_images), size=N_VOCAB_IMAGES, replace=False)

all_descs = []

for idx, i in enumerate(vocab_indices):

    d = extract_sift(train_images[i])

    if d is not None:

        all_descs.append(d)

    if idx % 1000 == 0:

        print(f'Vocabulary descriptors: {idx}/{N_VOCAB_IMAGES}')

all_descs = np.vstack(all_descs).astype(np.float32)

print(f'Total descriptors: {len(all_descs)}')

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

_, _, codebook = cv2.kmeans(all_descs, K, None, criteria, 3, cv2.KMEANS_RANDOM_CENTERS)

codebook = codebook.astype(np.float32)

print(f'Codebook shape: {codebook.shape}')

# BoVW encoding: assign each SIFT descriptor to its nearest codebook word,
# build a frequency histogram over the K words, then L2-normalise.
# Nearest-neighbor assignment uses the squared-L2 identity to avoid a large
# (N_desc x K x 128) intermediate tensor.
def encode_bovw(descs, codebook, K):

    if descs is None or len(descs) == 0:

        return np.zeros(K, dtype=np.float32)

    descs = descs.astype(np.float32)

    a2 = np.sum(descs ** 2, axis=1, keepdims=True)

    b2 = np.sum(codebook ** 2, axis=1, keepdims=True).T

    ab = descs @ codebook.T

    assignments = np.argmin(a2 + b2 - 2 * ab, axis=1)

    hist = np.bincount(assignments, minlength=K).astype(np.float32)

    hist = hist / (la.norm(hist) + 1e-7)

    return hist

feat_test = encode_bovw(extract_sift(train_images[0]), codebook, K)

assert feat_test.shape == (K,)

assert feat_test.dtype == np.float32

In [ ]:
k = 10

N_QUERIES = 100

TOTAL_RELEVANT = 5000

db_features_bovw = np.zeros((len(train_images), K), dtype=np.float32)

# Encode every training image as a BoVW histogram using the learned codebook
for i in range(len(train_images)):

    descs = extract_sift(train_images[i])

    db_features_bovw[i] = encode_bovw(descs, codebook, K)

    if i % 10000 == 0:

        print(f'Processed {i}/50000')

assert db_features_bovw.shape == (50000, K)

print(db_features_bovw.shape, db_features_bovw.dtype)

In [ ]:
# Retrieve top-k nearest neighbours by L2 distance
def retrieve(query_vec, db_features, k=10):

    dists = la.norm(db_features - query_vec, axis=1)

    top_k_idx = np.argsort(dists)[:k]

    return top_k_idx, dists[top_k_idx]

# Smoke test retrieval
query_vec = db_features_bovw[0]

top_k_idx, top_k_dists = retrieve(query_vec, db_features_bovw, k=10)

assert len(top_k_idx) == 10

assert top_k_dists[0] <= top_k_dists[-1]

print(top_k_idx, top_k_dists)

In [ ]:
# Fraction of retrieved results matching the query label
def precision_at_k(query_label, retrieved_labels, k):

    count = np.sum(retrieved_labels[:k] == query_label)

    return count / k

# Fraction of all relevant items retrieved in top-k
def recall_at_k(query_label, retrieved_labels, k, total_relevant):

    count = np.sum(retrieved_labels[:k] == query_label)

    return count / total_relevant

# Reciprocal rank of the first correct result
def reciprocal_rank(query_label, retrieved_labels):

    for i, label in enumerate(retrieved_labels):

        if label == query_label:

            return 1.0 / (i + 1)

    return 0.0

np.random.seed(42)

query_indices = np.random.choice(len(test_images), size=N_QUERIES, replace=False)

precisions = []

recalls = []

rr_scores = []

for qi in query_indices:

    q_img = test_images[qi]

    q_label = test_labels[qi]

    q_feat = encode_bovw(extract_sift(q_img), codebook, K)

    top_k_idx, _ = retrieve(q_feat, db_features_bovw, k=k)

    retrieved_labels = train_labels[top_k_idx]

    precisions.append(precision_at_k(q_label, retrieved_labels, k))

    recalls.append(recall_at_k(q_label, retrieved_labels, k, TOTAL_RELEVANT))

    rr_scores.append(reciprocal_rank(q_label, retrieved_labels))

mean_precision = np.mean(precisions)

mean_recall = np.mean(recalls)

mrr = np.mean(rr_scores)

print(f'BoVW | P@{k}: {mean_precision:.3f} | R@{k}: {mean_recall:.3f} | MRR: {mrr:.3f}')

print('Descriptor | P@k | R@k | MRR')

print(f'BoVW       | {mean_precision:.3f} | {mean_recall:.3f} | {mrr:.3f}')

In [ ]:
# Display query image and its top-k retrieved neighbours
def show_retrieval(query_img, query_label, top_k_imgs, top_k_labels, top_k_dists, class_names, k=10):

    fig, axes = plt.subplots(1, k + 1, figsize=(2 * (k + 1), 3))

    axes[0].imshow(query_img)

    axes[0].set_title('Query: ' + class_names[query_label])

    axes[0].axis('off')

    for j in range(k):

        axes[j + 1].imshow(top_k_imgs[j])

        axes[j + 1].set_title(class_names[top_k_labels[j]] + '\n' + str(round(top_k_dists[j], 3)))

        axes[j + 1].axis('off')

    plt.tight_layout()

    plt.show()

# Demo retrieval on a random test image
sample_idx = np.random.randint(len(test_images))

query_img = test_images[sample_idx]

query_label = test_labels[sample_idx]

query_feat = encode_bovw(extract_sift(query_img), codebook, K)

top_k_idx, top_k_dists = retrieve(query_feat, db_features_bovw, k=k)

top_k_imgs = train_images[top_k_idx]

top_k_labels = train_labels[top_k_idx]

show_retrieval(query_img, query_label, top_k_imgs, top_k_labels, top_k_dists, class_names, k=k)